In [ ]:
!pip install --user tqdm
import sys
sys.path.append('/hkfs/home/software/all/jupyter/base/2024-05-14/lib/python3.9/site-packages')

In [ ]:
import json
import re
import os
from tqdm import tqdm

# Read JSONL file and store data in the list
def read_jsonl(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return [json.loads(line) for line in file]

# Write JSONL file
def write_jsonl(file_path, data):
    with open(file_path, 'w', encoding='utf-8') as file:
        for entry in data:
            file.write(json.dumps(entry, ensure_ascii=False) + '\n')

# Create a dictionary of relational templates
def create_template_dict(templates):
    template_dict = {}
    for template in templates:
        template_dict[template["relation"]] = template["template"]
    return template_dict

# Extracting entities and relationships from triples
def parse_triple(triple):
    pattern = re.compile(r"\[entity1, [^\]]+\]: ([^,]+), \[relation, [^\]]+\]: ([^,]+), \[entity2, [^\]]+\]: ([^\]]+)")
    match = pattern.search(triple["triples"])
    """
    #unicode transform
    if match:
        entity1, relation, entity2 = match.groups()
        entity1 = entity1.encode().decode('unicode-escape')
        relation = relation.encode().decode('unicode-escape')
        entity2 = entity2.encode().decode('unicode-escape')
        return entity1, relation, entity2
    else:
        raise ValueError("Triple format is incorrect")
    """
    if match:
        entity1, relation, entity2 = match.groups()
        return entity1, relation, entity2
    else:
        raise ValueError("Triple format is incorrect")

# Generate sentences based on templates
def generate_sentences(triples, template_dict):
    sentences = []
    for triple in triples:
        try:
            entity1, relation, entity2 = parse_triple(triple)
            if relation in template_dict:
                sentence = template_dict[relation].replace("{entity1}", entity1).replace("{entity2}", entity2)
                formatted_sentence = f"text2KG:text:#{sentence}#"
                sentences.append(formatted_sentence)
            else:
                sentences.append(f"text2KG:text:#No template found for relation {relation}#")
        except ValueError as e:
            sentences.append(f"text2KG:text#{str(e)}#")
    return sentences

# Handling multiple JSONL files
def process_files(triples_dir, templates_file, output_dir):
    templates = read_jsonl(templates_file)
    template_dict = create_template_dict(templates)
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    files = [f for f in os.listdir(triples_dir) if f.endswith('.jsonl')]
    
    for file_name in tqdm(files, desc="Processing files"):
        file_path = os.path.join(triples_dir, file_name)
        triples = read_jsonl(file_path)
        sentences = generate_sentences(triples, template_dict)
        
        # Keep the output file name the same as the input file name
        output_file_name = file_name.replace('_reformed', '_sentences')
        output_file_path = os.path.join(output_dir, output_file_name)
        
        write_jsonl(output_file_path, sentences)

def main():
    triples_dir = '/hkfs/home/project/hk-project-p00201316/st_st176945/entity_rels_reformed'  # ternary group file directory
    templates_file = '/hkfs/home/project/hk-project-p00201316/st_st176945/relation_templates.jsonl'  #  template file
    output_dir = '/hkfs/home/project/hk-project-p00201316/st_st176945/generated_sentences'  # Directory of generated sentence files
    
    process_files(triples_dir, templates_file, output_dir)

if __name__ == "__main__":
    main()
